# Paradise, Mt Rainier: C- and L-band backscatter, summer 2026

Three radar backscatter datasets over the same slope of Mt Rainier, from the Paradise SNOTEL station (1563 m) up the Muir Snowfield to Camp Muir (3100 m), from mid-June to mid-October 2026:

- `sentinel1-rtc`: Sentinel-1 C-band (5.6 cm) RTC from Planetary Computer, 10 m;
- `opera-rtc`: the same Sentinel-1 passes, processed by OPERA, 30 m;
- `nisar-gcov`: NISAR L-band (24 cm) GCOV, 10 m. These are PROVISIONAL products, the only ones before 2026-10-21; `nisar-gcov` reads the VALIDATED ones unless asked.

All three are terrain-corrected gamma0 in linear power. Sentinel-1 images VV and VH here, NISAR HH and HV, so the two are compared by co-polarization (VV against HH) through gaia's common names. The stage file, [paradise.yaml](paradise.yaml), keeps the evening passes: ascending Sentinel-1 at about 19:00 PDT and NISAR track 172 (descending) at about 20:18 PDT, when the day's melt leaves the snow wet. OPERA and NISAR need `EARTHDATA_TOKEN`.

In [ ]:
OUTPUT = "/tmp/gaia-scratch/gaia-cli/paradise"  # a local folder or an s3:// URL
!gaia stage --config paradise.yaml -o {OUTPUT} --skip-existing
# `!` ignores the exit code; IPython keeps it in _exit_code
assert _exit_code == 0, "gaia failed; see its log above"  # noqa: F821

In [ ]:
import cf_xarray as cfxr
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rioxarray  # noqa: F401  registers the .rio accessor
import xarray as xr

from gaia_cli import io
from gaia_cli.datasets import cf_criteria

# ds.cf["gamma0_vv"]: Sentinel-1's vv; ds.cf["gamma0_hh"]: NISAR's HHHH
cfxr.set_options(custom_criteria=cf_criteria())

tree = io.open_datatree(f"{OUTPUT}/paradise-2026.zarr")
SNOTEL = (-121.7477, 46.7827)  # Paradise SNOTEL (679), longitude and latitude

copol = {
    "Sentinel-1 RTC (PC), VV": tree["rasters/sentinel1-rtc"].to_dataset().cf["gamma0_vv"],
    "OPERA RTC-S1, VV": tree["rasters/opera-rtc"].to_dataset().cf["gamma0_vv"],
    "NISAR GCOV, HH": tree["rasters/nisar-gcov"].to_dataset().cf["gamma0_hh"],
}
{name: da.sizes["time"] for name, da in copol.items()}

## Early and late summer

For each dataset, the mean of the passes in the last two weeks of June and in the second half of September, in dB, on each dataset's own grid. Sentinel-1's ascending passes come from two tracks (64 and 137), with slightly different viewing geometries; both are averaged here. NISAR's first pass is 26 June, so its June map is that one pass. The white dot is the SNOTEL station.

In June the Muir Snowfield, the upper half of each map, is wet and dark: liquid water in the snow absorbs the radar signal. By late September the snow has drained and refrozen or melted down to old firn and ice, and it is bright in both bands. Around Paradise, snow-free by mid-June, little changes.

In [ ]:
def db(da):
    return 10 * np.log10(da)


periods = {"late June": slice("2026-06-15", "2026-06-30"), "late September": slice("2026-09-15", "2026-09-30")}
fig, axes = plt.subplots(3, 3, figsize=(13, 13), layout="constrained")
for row, (name, da) in zip(axes, copol.items()):
    mean = {label: db(da.sel(time=span).mean("time")) for label, span in periods.items()}
    station = gpd.GeoSeries(gpd.points_from_xy(*[[v] for v in SNOTEL]), crs=4326).to_crs(da.rio.crs)
    for ax, (label, scene) in zip(row, mean.items()):
        scene.plot.imshow(ax=ax, cmap="gray", vmin=-25, vmax=0, cbar_kwargs={"label": "γ⁰ (dB)"})
        ax.set_title(f"{name}, {label}")
    (mean["late September"] - mean["late June"]).plot.imshow(
        ax=row[2], cmap="RdBu_r", vmin=-8, vmax=8, cbar_kwargs={"label": "change (dB)"}
    )
    row[2].set_title(f"{name}, September − June")
    for ax in row:
        station.plot(ax=ax, color="white", edgecolor="k", markersize=40)
        ax.set_aspect("equal")
        ax.set(xlabel="", ylabel="")
        ax.xaxis.set_major_locator(plt.MaxNLocator(3))

## Through the season

The median backscatter of every pass, on the snowfield (above 2000 m in the Copernicus DEM) and around Paradise (below 1800 m).

- On the snowfield, C-band VV brightens by about 4.5 dB from late June to October, most of it by mid-July. L-band HH brightens by less, about 2.7 dB, and more steadily. L-band penetrates further into wet snow, so the wet surface layer darkens it less.
- Around Paradise, NISAR HH is about 4 dB brighter than Sentinel-1 VV: a longer wavelength and a different polarization. On the snowfield the gap is about 3 dB in June and under 1 dB by October.
- The Planetary Computer and OPERA series are the same Sentinel-1 passes processed two ways. They agree to within about 0.5 dB, and OPERA averages 30 m cells. Their zigzag around Paradise is the two tracks' different viewing geometries.
- NISAR has no passes from 2026-07-27 to 2026-08-10, a gap in its instrument data.

In [ ]:
dem = tree["dem/cop30"].to_dataset().elevation
fig, (ax_hi, ax_lo) = plt.subplots(2, 1, figsize=(11, 7), sharex=True, layout="constrained")
for (name, da), color in zip(copol.items(), ["tab:blue", "tab:orange", "tab:green"]):
    elevation = dem.rio.reproject_match(da)  # onto each dataset's grid
    for ax, zone in ((ax_hi, elevation > 2000), (ax_lo, elevation < 1800)):
        series = db(da.where(zone).median(["x", "y"]))
        series.plot(ax=ax, marker="o", ms=4, lw=1, color=color, label=name)
ax_hi.set(title="Muir Snowfield (above 2000 m)", ylabel="γ⁰ (dB)", xlabel="")
ax_lo.set(title="Around Paradise (below 1800 m)", ylabel="γ⁰ (dB)", xlabel="")
ax_hi.legend();